In [1]:
import xarray as xr
import shutil

import os
from os.path import join
import glob
import numpy as np
import datetime

from joblib import Parallel, delayed
import joblib

from glob import glob

from functools import partial
import dask.array as da

import pandas as pd
import tqdm
import pickle

from pathlib import Path

from credit.pbs import get_num_cpus

In [2]:
top_dir = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset"
channels = [4, 7, 8, 9, 10, 13]
zarr_path = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km_2025.zarr"
year = 2025
save_dir = join(top_dir, str(year))

In [3]:
outfile = r"/glade/derecho/scratch/dkimpara/goes-cloud-dataset/intermediate_files/file_valid_time_dict_2025_final.pkl"
with open(outfile, "rb") as handle:
    valid_time_dict = pickle.load(handle)

sorted_files = sorted(valid_time_dict.keys())

In [ ]:
zarr_ds = xr.open_dataset(zarr_path, consolidated=False)
zarr_times = zarr_ds.t
assert np.all(zarr_times[:-1] <= zarr_times[1:])

In [5]:
def sel_within_tolerance(ds, time_points, tolerance, dim="t"):
    """
    Select time points from ds within a tolerance, dropping any that don't match.
    
    Parameters:
        ds         : xarray Dataset or DataArray
        time_points: list of times to select
        tolerance  : pd.Timedelta or np.timedelta64
        dim        : time dimension name (default "time")
    
    Returns:
        Dataset/DataArray with only the matched selections
    """
    valid_times = []
    for t in time_points:
        nearest = ds[dim].sel({dim: t}, method="nearest")
        if abs(nearest.values - np.datetime64(t)) <= tolerance:
            valid_times.append(nearest.values)
    
    return ds.sel({dim: valid_times})

# write to zarr

- sorted_files are ordered

In [ ]:
def write_zarr(chunk):
    for file in chunk:
        time_str = Path(file).name[:-7]
        paths = [join(Path(file).parent ,f"{time_str}_C{channel:02}.nc") for channel in channels]
    
        datasets = [xr.open_dataset(p).sortby("t") for p in paths]
    
        valid_times = valid_time_dict[file]
        datasets = [sel_within_tolerance(ds, valid_times, pd.Timedelta(1, "m")).sortby("t") for ds in datasets]
    
        ds_comb = xr.concat(datasets,
                            dim="channel",
                            join="override")
    
        ds_comb = ds_comb[["BT_or_R", "yaw_flip_flag", "BT_or_R_mean"]].rename({"lat": "latitude", "lon": "longitude"})
        ds_comb = ds_comb.drop_vars(['x_image', 'y_image', 'latitude_longitude', "latitude", "longitude", "channel" ])
    
        ds_comb = ds_comb.chunk({"t": 1})
    
        first_time = min(valid_times)
        index = np.searchsorted(zarr_times, first_time)
        # print(index)
        if index < 0:
            print('error')
    
        end_index = index + len(valid_times)
    
        ds_comb.to_zarr(zarr_path,
                        region={"t": slice(index, end_index),},
                        align_chunks=True, consolidated=False)

In [ ]:
num_cpus = get_num_cpus()
# num_cpus = 2
chunked = np.array_split(sorted_files, 100 * (num_cpus - 1))

res = Parallel(n_jobs=num_cpus - 1)(delayed(write_zarr)(chunk) for chunk in chunked)